# Embedding & Indexing: Both Research Papers (PyMuPDF4LLM)

Same pipeline as **Embedding_Indexing_PDF1** except **loading**: this notebook uses **PyMuPDF4LLM** to extract **all information** from **both** research papers (e.g. from `./input_docs` or `./ResearchPapers`), then chunks, embeds, and indexes in Pinecone for chat.

**Steps:**
1. **Load both PDFs** with PyMuPDF4LLM (page_chunks, full metadata: tables, images, text).
2. Split into chunks, optionally enrich with LLM context.
3. Generate embeddings and index in Pinecone.
4. Chat-based retriever over both papers.

## 1. Install dependencies

In [1]:
!pip install -q gdown pdfplumber pymupdf4llm pymupdf
!pip install -q -U langchain-pinecone langchain-openai langchain langchain-community langchain-core
!pip install -q -U langchain-classic
!pip install -q -U "unstructured[all-docs]" sentence-transformers python-dotenv openai pinecone langchain-classic

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-pinecone 0.2.13 requires pinecone[asyncio]<8.0.0,>=6.0.0, but you have pinecone 8.1.0 which is incompatible.


## 2. Environment and embedding setup

Set `PINECONE_API_KEY` and `OPENAI_API_KEY` in a `.env` file or in your environment.

In [2]:
import os
try:
    from google.colab import userdata
    OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')
    PINECONE_API_KEY = userdata.get('PINECONE_API_KEY')
except Exception:
    from dotenv import load_dotenv
    load_dotenv()
    OPENAI_API_KEY = os.environ.get('OPENAI_API_KEY') or 'your-openai-key'
    PINECONE_API_KEY = os.environ.get('PINECONE_API_KEY') or 'your-pinecone-key'
print(f"OPENAI_API_KEY: {OPENAI_API_KEY[:7]} .....{OPENAI_API_KEY[-7:]}")
print(f"PINECONE_API_KEY: {PINECONE_API_KEY[:7]} .....{PINECONE_API_KEY[-7:]}")
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
os.environ["PINECONE_API_KEY"] = PINECONE_API_KEY

OPENAI_API_KEY: sk-proj .....2RBkeMA
PINECONE_API_KEY: pcsk_6S .....is99vuV


In [3]:
import os
from pathlib import Path
from dotenv import load_dotenv
from pinecone import Pinecone, ServerlessSpec
from langchain_pinecone import PineconeVectorStore
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
try:
    from langchain_classic.chains import ConversationalRetrievalChain
    from langchain_classic.memory import ConversationSummaryMemory
except ImportError:
    ConversationalRetrievalChain = None
    ConversationSummaryMemory = None

pc = Pinecone(api_key=PINECONE_API_KEY)
EMBEDDING_MODEL = "text-embedding-ada-002"
embeddings = OpenAIEmbeddings(model=EMBEDDING_MODEL)
EMBEDDING_DIMENSION = 1536
print("Pinecone and OpenAI embeddings initialized.")

Pinecone and OpenAI embeddings initialized.


## 3. Create or connect to Pinecone index

In [4]:
INDEX_NAME = "pdf-embedding-index"
METRIC = "cosine"
existing = pc.list_indexes()
if INDEX_NAME not in [idx.name for idx in existing]:
    pc.create_index(
        name=INDEX_NAME,
        dimension=EMBEDDING_DIMENSION,
        metric=METRIC,
        spec=ServerlessSpec(cloud="aws", region="us-east-1"),
    )
    print(f"Created index '{INDEX_NAME}'.")
else:
    print(f"Using existing index '{INDEX_NAME}'.")
index = pc.Index(INDEX_NAME)
print(index.describe_index_stats())

Created index 'pdf-embedding-index'.
{'_response_info': {'raw_headers': {'connection': 'keep-alive',
                                    'content-length': '151',
                                    'content-type': 'application/json',
                                    'date': 'Thu, 05 Mar 2026 21:05:35 GMT',
                                    'grpc-status': '0',
                                    'server': 'envoy',
                                    'x-envoy-upstream-service-time': '40',
                                    'x-pinecone-request-latency-ms': '39',
                                    'x-pinecone-response-duration-ms': '42'}},
 'dimension': 1536,
 'index_fullness': 0.0,
 'memoryFullness': 0.0,
 'metric': 'cosine',
 'namespaces': {},
 'storageFullness': 0.0,
 'total_vector_count': 0,
 'vector_type': 'dense'}


## 4. Load both research papers (PyMuPDF4LLM) and chunk

**Loading module from test_loader_research_papers:** PyMuPDF4LLM extracts full text (Markdown), tables, and image metadata per page. All PDFs in `INPUT_DIR` or `ResearchPapers` are loaded and combined into `documents`.

In [5]:
import pymupdf4llm

INPUT_DIR = "./input_docs"
all_pdfs = []
for d in ("./input_docs", "./ResearchPapers"):
    if os.path.isdir(d):
        all_pdfs = sorted(Path(d).glob("*.pdf"))
        if all_pdfs:
            INPUT_DIR = d
            break
if not all_pdfs and os.path.isdir(INPUT_DIR):
    all_pdfs = sorted(Path(INPUT_DIR).glob("*.pdf"))
if not all_pdfs:
    raise FileNotFoundError(f"No PDFs in {INPUT_DIR}. Add research papers to run.")

documents = []
for pdf_path in all_pdfs:
    path_str = str(pdf_path)
    print(f"Loading {pdf_path.name}...")
    page_chunks = pymupdf4llm.to_markdown(
        path_str,
        page_chunks=True,
        table_strategy="lines",
        show_progress=False,
    )
    for i, chunk in enumerate(page_chunks):
        meta = dict(chunk.get("metadata", {}))
        page_num = meta.get("page_number") or meta.get("page") or (i + 1)
        meta["page_number"] = page_num
        meta["source"] = path_str
        meta["page"] = page_num
        text = (chunk.get("text") or "").strip()
        tables = chunk.get("tables") or []
        images = chunk.get("images") or []
        meta["total_pages"] = len(page_chunks)
        meta["tables_on_page"] = len(tables)
        meta["images_on_page"] = len(images)
        doc = Document(page_content=text, metadata=meta)
        documents.append(doc)
    print(f"  -> {len(page_chunks)} pages")

print(f"Total: {len(documents)} page-level documents from {len(all_pdfs)} PDF(s).")

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=150,
    length_function=len,
)
chunks = text_splitter.split_documents(documents)
print(f"Split into {len(chunks)} chunks.")

Consider using the pymupdf_layout package for a greatly improved page layout analysis.
Loading FasterR_CNN.pdf...
  -> 17 pages
Loading GradientDescent.pdf...
  -> 14 pages
Total: 31 page-level documents from 2 PDF(s).
Split into 201 chunks.


## 4b. Contextual retrieval: enrich chunks with LLM-generated context

Each chunk is sent with document context to an LLM to produce a short summary; that context is prepended to the chunk before indexing.

In [6]:
from langchain_openai import ChatOpenAI

full_doc_text = "\n\n".join(d.page_content for d in documents)[:8000]
augmented_chunks = []
if chunks and OPENAI_API_KEY:
    print("Contextual enrichment: chunk + document → LLM → Context + Chunk...")
    llm_ctx = ChatOpenAI(temperature=0, model="gpt-4.1", openai_api_key=OPENAI_API_KEY)
    prompt = """Given this document excerpt and a chunk from it, in 1–2 sentences summarize what the chunk is about and when it would be relevant to a search.

Document excerpt:
{document}

Chunk:
{chunk}

Context (1–2 sentences):"""
    for i, doc in enumerate(chunks):
        try:
            r = llm_ctx.invoke(prompt.format(document=full_doc_text[:3000], chunk=doc.page_content[:700]))
            ctx = (r.content if hasattr(r, "content") else str(r)).strip()
            augmented_chunks.append(Document(page_content=f"[Context: {ctx}]\n\n{doc.page_content}", metadata=doc.metadata))
        except Exception:
            augmented_chunks.append(doc)
        if (i + 1) % 15 == 0:
            print(f"  Enriched {i+1}/{len(chunks)} chunks.")
    print(f"Created {len(augmented_chunks)} context-augmented chunks.")
else:
    augmented_chunks = chunks
    print("Skipping contextual enrichment (no OPENAI_API_KEY or no chunks). Using raw chunks.")

Contextual enrichment: chunk + document → LLM → Context + Chunk...
  Enriched 15/201 chunks.
  Enriched 30/201 chunks.
  Enriched 45/201 chunks.
  Enriched 60/201 chunks.
  Enriched 75/201 chunks.
  Enriched 90/201 chunks.
  Enriched 105/201 chunks.
  Enriched 120/201 chunks.
  Enriched 135/201 chunks.
  Enriched 150/201 chunks.
  Enriched 165/201 chunks.
  Enriched 180/201 chunks.
  Enriched 195/201 chunks.
Created 201 context-augmented chunks.


## 5. Generate embeddings and index in Pinecone

In [9]:
os.environ["PINECONE_API_KEY"] = PINECONE_API_KEY

def _sanitize_meta(meta):
    """Pinecone accepts only string, number, boolean, or list of strings. Remove None and convert invalid types."""
    out = {}
    for k, v in (meta or {}).items():
        if v is None:
            out[k] = ""
        elif isinstance(v, list):
            out[k] = [str(x) if x is not None else "" for x in v]
        elif isinstance(v, (str, int, float, bool)):
            out[k] = v
        else:
            out[k] = str(v)
    return out

docs_to_index = augmented_chunks if augmented_chunks else chunks
docs_sanitized = [Document(page_content=d.page_content, metadata=_sanitize_meta(d.metadata)) for d in docs_to_index]
vectorstore = PineconeVectorStore.from_documents(
    documents=docs_sanitized,
    embedding=embeddings,
    index_name=INDEX_NAME,
)
print(f"Indexed {len(docs_to_index)} chunks into Pinecone index '{INDEX_NAME}'.")
import time
time.sleep(3)
stats = pc.Index(INDEX_NAME).describe_index_stats()
print("Index stats:", stats)

Indexed 201 chunks into Pinecone index 'pdf-embedding-index'.
Index stats: {'_response_info': {'raw_headers': {'connection': 'keep-alive',
                                    'content-length': '186',
                                    'content-type': 'application/json',
                                    'date': 'Thu, 05 Mar 2026 21:16:12 GMT',
                                    'grpc-status': '0',
                                    'server': 'envoy',
                                    'x-envoy-upstream-service-time': '43',
                                    'x-pinecone-request-latency-ms': '42',
                                    'x-pinecone-response-duration-ms': '45'}},
 'dimension': 1536,
 'index_fullness': 0.0,
 'memoryFullness': 0.0,
 'metric': 'cosine',
 'namespaces': {'__default__': {'vector_count': 201}},
 'storageFullness': 0.0,
 'total_vector_count': 201,
 'vector_type': 'dense'}


## 6. Chat-based retriever

Conversational retrieval over both papers using chat history and context-enriched chunks.

In [10]:
from langchain_core.prompts import PromptTemplate

file_names = list(set([doc.metadata.get('source') for doc in documents]))
file_list_str = ", ".join([Path(f).name for f in file_names])
print("Files in context:", file_list_str)
template = f"""You are a helpful assistant. You are analyzing the following specific files: {file_list_str}. Use the following pieces of context to answer the question at the end. You will understand the user's context based on the chat history, documents that have been provided and then only answer the question.
If you don't know the answer, just say that you don't know, don't try to make up an answer.

Context: {{context}}

Question: {{question}}
Helpful Answer:"""
QA_CHAIN_PROMPT = PromptTemplate.from_template(template)

Files in context: FasterR_CNN.pdf, GradientDescent.pdf


In [15]:
llm = ChatOpenAI(temperature=0.0, model="gpt-4.1", openai_api_key=OPENAI_API_KEY)
memory = ConversationSummaryMemory(llm=llm, memory_key="chat_history", return_messages=True, output_key="answer") if ConversationSummaryMemory else None

if vectorstore and ConversationalRetrievalChain:
    retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 3})
    qa_chain = ConversationalRetrievalChain.from_llm(
        llm=llm,
        retriever=retriever,
        memory=memory,
        return_source_documents=True,
        verbose=False,
        combine_docs_chain_kwargs={"prompt": QA_CHAIN_PROMPT}
    )
    print("Conversational Retrieval Chain created.")
else:
    qa_chain = None
    print("Vectorstore or ConversationalRetrievalChain not available.")

chat_history = []

def chat_with_rag_pipeline(query: str):
    global chat_history
    if not qa_chain:
        print("QA chain not initialized.")
        return "Error: QA chain not set up."
    try:
        print(f"\nUser Query: {query}")
        result = qa_chain.invoke({"question": query, "chat_history": chat_history})
        answer = result["answer"]
        print(f"LLM Answer: {answer}")
        chat_history.append((query, answer))
        print("\nSources Used:")
        if result.get("source_documents"):
            for i, doc in enumerate(result["source_documents"]):
                src = doc.metadata.get('source', 'Unknown')
                preview = doc.page_content[:150] + "..." if len(doc.page_content) > 150 else doc.page_content
                print(f"  {i+1}. {Path(src).name}\n     {preview}\n")
        else:
            print("  No source documents returned.")
        return answer
    except Exception as e:
        print(f"Error: {e}")
        return str(e)

if chunks and qa_chain:
    print("\n--- Starting Chat Session ---")
    # First question
    question1 = "What are the main topics covered in the provided documents?"
    response1 = chat_with_rag_pipeline(question1)

    # Follow-up question (demonstrating chat history usage)
    if "Error:" not in response1: # Proceed only if first question was successful
        question2 = "Can you elaborate on one of those topics? Pick one you find interesting."
        response2 = chat_with_rag_pipeline(question2)

    # Follow-up question (demonstrating chat history usage)
    if "Error:" not in response2: # Proceed only if first question was successful
        question3 = "Tell me more about the first pdf file that was loaded"
        response3 = chat_with_rag_pipeline(question3)

    # Follow-up question (demonstrating chat history usage)
    if "Error:" not in response3: # Proceed only if first question was successful
        question4 = "Tell me more today's weather"
        response4 = chat_with_rag_pipeline(question4)

     # Follow-up question (demonstrating chat history usage)
    if "Error:" not in response4: # Proceed only if first question was successful
        question5 = "Tell me more SSD"
        response5 = chat_with_rag_pipeline(question5)
else:
    print("\nSkipping chat example as no documents were processed or QA chain is not set up.")
    print("Please add documents to the 'input_docs' folder and re-run the notebook if you wish to chat.")

Conversational Retrieval Chain created.

--- Starting Chat Session ---

User Query: What are the main topics covered in the provided documents?
LLM Answer: Based on the provided context, the main topics covered in the documents are:

1. **Faster R-CNN and Object Detection Architectures**  
   - The references and context mention foundational works related to object detection, including Faster R-CNN, SSD (Single Shot MultiBox Detector), Caffe, and other related deep learning architectures.  
   - Topics include the development, evaluation, and error analysis of object detectors, as well as specific applications like pedestrian detection.

2. **Gradient Descent Optimization Algorithms**  
   - The summary from GradientDescent.pdf indicates a focus on explaining various algorithms for optimizing gradient descent, including their variants and the challenges encountered during training neural networks.
   - The document covers common optimization methods, their motivations, derivation of up

## 7. Deploy with Gradio

Run the cell below to install Gradio and launch a chat UI. Use **Share** to get a public link (e.g. for [Hugging Face Spaces](https://huggingface.co/spaces) or local sharing).

In [16]:
!pip install -q gradio

In [17]:
import gradio as gr

def rag_chat(message, history):
    """Gradio chat handler: uses qa_chain and returns only the assistant reply (no console print)."""
    if not qa_chain:
        return "Error: QA chain not set up. Run the cells above (load PDFs, index, create chain) first."
    try:
        result = qa_chain.invoke({"question": message, "chat_history": chat_history})
        answer = result.get("answer", "")
        chat_history.append((message, answer))
        # Optionally append source names to the reply
        if result.get("source_documents"):
            sources = list(set(d.metadata.get("source", "") for d in result["source_documents"]))
            source_names = ", ".join(Path(s).name for s in sources if s)
            if source_names:
                answer += "\n\n---\n*Sources: " + source_names + "*"
        return answer
    except Exception as e:
        return f"Error: {e}"

demo = gr.ChatInterface(
    fn=rag_chat,
    title="RAG Chat: Research Papers",
    description="Ask questions about your indexed PDFs (Faster R-CNN, Gradient Descent, etc.). Run all cells above first.",
)

if qa_chain:
    demo.launch(share=True)
else:
    print("Run the previous cells to build qa_chain, then re-run this cell.")

TypeError: ChatInterface.__init__() got an unexpected keyword argument 'type'

## (Optional) Delete Pinecone index

Run this only if you want to remove the index and free resources.

In [ ]:
try:
    pc.delete_index(INDEX_NAME)
    print(f"Index '{INDEX_NAME}' deleted successfully.")
except Exception as e:
    print(f"Could not delete index: {e}")